# Full Video Matting + Video Background Pipeline

**Run all cells top to bottom. Do NOT skip any cell.**

| Section | What it does |
|---|---|
| 1 | Install dependencies |
| 2 | Download background video (Mixkit, free) |
| 3 | DeepLabV3 segmentation (Option A — no model download needed) |
| 4 | RVM segmentation (Option B — higher quality, needs model file) |
| 5 | Load Stable Diffusion for frame refinement |
| 6 | **VIDEO background composite** with anti-flicker (NEW) |
| 7 | Encode final MP4 |

> **Run either Section 3 OR Section 4** for segmentation — not both.
> RVM (Section 4) gives better edges but requires downloading `rvm_mobilenetv3.pth`.

---
## SECTION 1 — Install Dependencies

In [ ]:
!pip install opencv-python tqdm -q

In [ ]:
!pip install -q "Pillow==10.4.0" --force-reinstall
# import os; os.kill(os.getpid(), 9)   # restart runtime after install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.0 MB/s eta 0:00:00


In [ ]:
import PIL
import torch
import torchvision
print("Pillow:", PIL.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

Pillow: 11.3.0
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128


In [ ]:
!pip install -q torch torchvision
!git clone https://github.com/PeterL1n/RobustVideoMatting.git
import sys; sys.path.insert(0, '/content/RobustVideoMatting')

Cloning into 'RobustVideoMatting'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 9.00 MiB | 13.33 MiB/s, done.
Resolving deltas: 100% (81/81), done.


---
## SECTION 2 - Segmentation: Option A (DeepLabV3)
No model download needed. Works out of the box.
Edges are softer/less accurate than RVM.

> **Skip this section if you want to use RVM (Section 3)**

In [ ]:
# VIDEO_PATH  = "/content/src.mp4"
# OUTPUT_DIR  = "/content/output"

# BATCH_SIZE  = 8      # frames processed at once — increase if GPU has more memory
#                      # T4 (Colab free): 8-16 is safe
#                      # A100 (Colab Pro): 32-64

# BLUR_KSIZE  = 9      # edge softness (0 = hard edges)
# RESIZE      = None   # e.g. (1920, 1080) or None to keep original
# FMT         = "png"  # "png" or "jpg"
# START_FRAME = 0
# END_FRAME   = None   # None = all frames


# # Imports & model

# import cv2
# import numpy as np
# from pathlib import Path
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor
# import torch
# import torch.nn.functional as F
# import torchvision.transforms.functional as TF
# from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device : {DEVICE}")
# if DEVICE == "cpu":
#     print("WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU")

# # Load model
# weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
# model   = deeplabv3_resnet50(weights=weights).to(DEVICE)
# model.eval()

# # Use half precision on GPU for ~2x speed boost
# if DEVICE == "cuda":
#     model = model.half()
#     print("Using FP16 (half precision) for faster inference.")

# print("Model ready.\n")

# PERSON_CLASS = 15  # COCO person class

# # ImageNet normalization constants as tensors (on device)
# MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
# STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
# if DEVICE == "cuda":
#     MEAN = MEAN.half()
#     STD  = STD.half()

# # Core batch segmentation

# def frames_to_tensor(frames_bgr):
#     """Convert a list of BGR numpy frames to a normalized GPU tensor (B, 3, H, W)."""
#     tensors = []
#     for f in frames_bgr:
#         rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
#         t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0  # (3, H, W)
#         tensors.append(t)
#     batch = torch.stack(tensors).to(DEVICE)   # (B, 3, H, W)
#     if DEVICE == "cuda":
#         batch = batch.half()
#     batch = (batch - MEAN) / STD
#     return batch


# def segment_batch(frames_bgr, blur_ksize=9):
#     """
#     Segment a batch of BGR frames at once on GPU.

#     Returns:
#         fgr_list : list of (H, W, 3) uint8 — person in color, bg black
#         pha_list : list of (H, W)    uint8 — white=person, black=bg
#     """
#     h, w = frames_bgr[0].shape[:2]

#     with torch.no_grad():
#         batch  = frames_to_tensor(frames_bgr)              # (B, 3, H, W)
#         out    = model(batch)["out"]                       # (B, 21, H, W)
#         preds  = out.argmax(dim=1).byte()                  # (B, H, W)
#         masks  = (preds == PERSON_CLASS).float()           # (B, H, W) 0.0/1.0

#     masks_np = (masks.cpu().numpy() * 255).astype(np.uint8)  # (B, H, W)

#     fgr_list, pha_list = [], []
#     for i, frame in enumerate(frames_bgr):
#         binary = masks_np[i]

#         # Soften edges
#         if blur_ksize > 1:
#             k   = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1
#             pha = cv2.GaussianBlur(binary, (k, k), 0)
#         else:
#             pha = binary

#         mask_f = pha.astype(np.float32) / 255.0
#         fgr    = np.clip(frame.astype(np.float32) * mask_f[..., None], 0, 255).astype(np.uint8)

#         fgr_list.append(fgr)
#         pha_list.append(pha)

#     return fgr_list, pha_list



# # Fast parallel save

# def save_frame(args):
#     """Save a single fgr + pha pair. Runs in a thread pool."""
#     fgr, pha, fgr_path, pha_path, fmt = args
#     png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
#     jpg_params = [cv2.IMWRITE_JPEG_QUALITY, 95]
#     cv2.imwrite(fgr_path, fgr, png_params if fmt == "png" else jpg_params)
#     cv2.imwrite(pha_path, pha)


# # Main loop

# def preprocess_video(video_path, output_dir, batch_size=8, blur_ksize=9,
#                      resize=None, fmt="png", start_frame=0, end_frame=None):

#     fgr_dir = Path(output_dir) / "fgr"
#     pha_dir = Path(output_dir) / "pha"
#     fgr_dir.mkdir(parents=True, exist_ok=True)
#     pha_dir.mkdir(parents=True, exist_ok=True)

#     cap = cv2.VideoCapture(str(video_path))
#     if not cap.isOpened():
#         raise RuntimeError(f"Cannot open video: {video_path}")

#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     fps          = cap.get(cv2.CAP_PROP_FPS)
#     width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     end_frame    = end_frame or total_frames

#     print(f"Video      : {video_path}")
#     print(f"Resolution : {width}x{height}  |  FPS: {fps:.2f}  |  Frames: {total_frames}")
#     print(f"Processing : {start_frame} → {end_frame}  ({end_frame - start_frame} frames)")
#     print(f"Batch size : {batch_size}  |  Device: {DEVICE}")
#     print(f"Output     : {output_dir}")
#     print()

#     if start_frame > 0:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

#     saved      = 0
#     frame_buf  = []   # buffer of raw BGR frames for current batch

#     # Thread pool for parallel disk writes (so GPU isn't waiting on I/O)
#     executor = ThreadPoolExecutor(max_workers=4)
#     futures  = []

#     try:
#         with tqdm(total=end_frame - start_frame, desc="Segmenting", unit="fr",
#                   dynamic_ncols=True) as pbar:

#             for global_idx in range(start_frame, end_frame):
#                 ret, frame = cap.read()
#                 if not ret:
#                     break

#                 if resize:
#                     frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LANCZOS4)

#                 frame_buf.append(frame)

#                 # Process when buffer is full OR we're on the last frame
#                 if len(frame_buf) == batch_size or global_idx == end_frame - 1:
#                     fgr_list, pha_list = segment_batch(frame_buf, blur_ksize)

#                     # Submit saves to thread pool (non-blocking)
#                     for fgr, pha in zip(fgr_list, pha_list):
#                         fgr_path = str(fgr_dir / f"{saved:05d}.{fmt}")
#                         pha_path = str(pha_dir  / f"{saved:05d}.png")
#                         futures.append(
#                             executor.submit(save_frame,
#                                             (fgr, pha, fgr_path, pha_path, fmt))
#                         )
#                         saved += 1

#                     pbar.update(len(frame_buf))
#                     frame_buf = []

#         # Wait for all saves to finish
#         for f in futures:
#             f.result()

#     finally:
#         cap.release()
#         executor.shutdown(wait=True)

#     print(f"\n Done!  Saved {saved} frames.")
#     print(f"  fgr → {fgr_dir}")
#     print(f"  pha → {pha_dir}")


# # Run

# preprocess_video(
#     video_path  = VIDEO_PATH,
#     output_dir  = OUTPUT_DIR,
#     batch_size  = BATCH_SIZE,
#     blur_ksize  = BLUR_KSIZE,
#     resize      = RESIZE,
#     fmt         = FMT,
#     start_frame = START_FRAME,
#     end_frame   = END_FRAME,
# )

---
## SECTION 3 - Segmentation: Option B (RVM - Better Quality)
Requires `rvm_mobilenetv3.pth` uploaded to `/content/`.
Download from: https://github.com/PeterL1n/RobustVideoMatting/releases

> **Skip this section if you already ran Section 3 (DeepLabV3)**

In [ ]:
VIDEO_PATH  = "/content/src.mp4"
OUTPUT_DIR  = "/content/output"

BATCH_SIZE  = 8      # frames processed at once — increase if GPU has more memory
                     # T4 (Colab free): 8-16 is safe
                     # A100 (Colab Pro): 32-64

BLUR_KSIZE  = 9      # edge softness (0 = hard edges)
RESIZE      = None   # e.g. (1920, 1080) or None to keep original
FMT         = "png"  # "png" or "jpg"
START_FRAME = 0
END_FRAME   = None   # None = all frames

# Load RVM instead of DeepLabV3

import torch
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from model import MattingNetwork   # from the cloned repo

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

# Load RVM with MobileNetV3 backbone (fastest, good quality)
# Alternative backbone: 'resnet50' — slower but higher quality
model = MattingNetwork('mobilenetv3').eval().to(DEVICE)
model.load_state_dict(
    torch.load('/content/rvm_mobilenetv3.pth', map_location=DEVICE)
)
if DEVICE == "cuda":
    model = model.half()
    print("Using FP16.")

print("RVM ready.\n")

# RVM carries its own recurrent state between frames — init once per video
rec = [None] * 4   # r1, r2, r3, r4  (hidden states for the GRU)


# New segment_batch using RVM


def frames_to_tensor_rvm(frames_bgr):
    """BGR list → normalized float tensor (B, 3, H, W) for RVM."""
    tensors = []
    for f in frames_bgr:
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        tensors.append(t)
    batch = torch.stack(tensors).to(DEVICE)
    if DEVICE == "cuda":
        batch = batch.half()
    return batch   # RVM normalizes internally — no manual mean/std needed


def segment_batch(frames_bgr, blur_ksize=9):
    """
    RVM-based matting. Replaces DeepLabV3 + guided filter entirely.
    rec[] carries temporal state so edges stay stable across frames.
    blur_ksize is kept for API compatibility but no longer used.
    """
    global rec

    with torch.no_grad():
        src = frames_to_tensor_rvm(frames_bgr)    # (B, 3, H, W)

        # RVM processes one frame at a time to thread the recurrent state
        fgr_tensors, pha_tensors = [], []
        for i in range(src.shape[0]):
            frame_t = src[i:i+1]                  # (1, 3, H, W)
            fgr_t, pha_t, *rec = model(frame_t, *rec, downsample_ratio=0.25)
            fgr_tensors.append(fgr_t)
            pha_tensors.append(pha_t)

        fgr_batch = torch.cat(fgr_tensors)        # (B, 3, H, W)  float [0,1]
        pha_batch = torch.cat(pha_tensors)        # (B, 1, H, W)  float [0,1]

    fgr_np  = (fgr_batch.float().cpu().numpy() * 255).astype(np.uint8)
    pha_np  = (pha_batch.float().cpu().numpy() * 255).astype(np.uint8)

    fgr_list, pha_list = [], []
    for i in range(len(frames_bgr)):
        # fgr: (3, H, W) → BGR (H, W, 3)
        fgr = cv2.cvtColor(fgr_np[i].transpose(1, 2, 0), cv2.COLOR_RGB2BGR)
        pha = pha_np[i, 0]                        # (H, W)
        fgr_list.append(fgr)
        pha_list.append(pha)

    return fgr_list, pha_list

# Fast parallel save

def save_frame(args):
    """Save a single fgr + pha pair. Runs in a thread pool."""
    fgr, pha, fgr_path, pha_path, fmt = args
    png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
    jpg_params = [cv2.IMWRITE_JPEG_QUALITY, 95]
    cv2.imwrite(fgr_path, fgr, png_params if fmt == "png" else jpg_params)
    cv2.imwrite(pha_path, pha)


# Main loop

global rec; rec = [None] * 4

def preprocess_video(video_path, output_dir, batch_size=8, blur_ksize=9,
                     resize=None, fmt="png", start_frame=0, end_frame=None):

    fgr_dir = Path(output_dir) / "fgr"
    pha_dir = Path(output_dir) / "pha"
    fgr_dir.mkdir(parents=True, exist_ok=True)
    pha_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    end_frame    = end_frame or total_frames

    print(f"Video      : {video_path}")
    print(f"Resolution : {width}x{height}  |  FPS: {fps:.2f}  |  Frames: {total_frames}")
    print(f"Processing : {start_frame} → {end_frame}  ({end_frame - start_frame} frames)")
    print(f"Batch size : {batch_size}  |  Device: {DEVICE}")
    print(f"Output     : {output_dir}")
    print()

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    saved      = 0
    frame_buf  = []   # buffer of raw BGR frames for current batch

    # Thread pool for parallel disk writes (so GPU isn't waiting on I/O)
    executor = ThreadPoolExecutor(max_workers=4)
    futures  = []

    try:
        with tqdm(total=end_frame - start_frame, desc="Segmenting", unit="fr",
                  dynamic_ncols=True) as pbar:

            for global_idx in range(start_frame, end_frame):
                ret, frame = cap.read()
                if not ret:
                    break

                if resize:
                    frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LANCZOS4)

                frame_buf.append(frame)

                # Process when buffer is full OR we're on the last frame
                if len(frame_buf) == batch_size or global_idx == end_frame - 1:
                    fgr_list, pha_list = segment_batch(frame_buf, blur_ksize)

                    # Submit saves to thread pool (non-blocking)
                    for fgr, pha in zip(fgr_list, pha_list):
                        fgr_path = str(fgr_dir / f"{saved:05d}.{fmt}")
                        pha_path = str(pha_dir  / f"{saved:05d}.png")
                        futures.append(
                            executor.submit(save_frame,
                                            (fgr, pha, fgr_path, pha_path, fmt))
                        )
                        saved += 1

                    pbar.update(len(frame_buf))
                    frame_buf = []

        # Wait for all saves to finish
        for f in futures:
            f.result()

    finally:
        cap.release()
        executor.shutdown(wait=True)

    print(f"\n Done!  Saved {saved} frames.")
    print(f"  fgr → {fgr_dir}")
    print(f"  pha → {pha_dir}")


#  Run

preprocess_video(
    video_path  = VIDEO_PATH,
    output_dir  = OUTPUT_DIR,
    batch_size  = BATCH_SIZE,
    blur_ksize  = BLUR_KSIZE,
    resize      = RESIZE,
    fmt         = FMT,
    start_frame = START_FRAME,
    end_frame   = END_FRAME,
)

Device : cuda
Using FP16.
RVM ready.

Video      : /content/src.mp4
Resolution : 3840x2160  |  FPS: 29.97  |  Frames: 416
Processing : 0 → 416  (416 frames)
Batch size : 8  |  Device: cuda
Output     : /content/output



Segmenting:  92%|█████████▏| 384/416 [02:09<00:10,  2.97fr/s]



 Done!  Saved 384 frames.
  fgr → /content/output/fgr
  pha → /content/output/pha


---
## SECTION 5 - Load Stable Diffusion (optional refinement)
This loads the diffusion model used to stylize frames.
Skip sections 5a/5b if you don't want diffusion refinement.

In [ ]:
!pip install -q diffusers transformers accelerate xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.2 MB/s eta 0:00:00


In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import cv2
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(DEVICE)

# IMPORTANT for Colab (reduces VRAM)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_attention_slicing()

pipe.safety_checker = None

# Fixed seed → no flicker
generator = torch.Generator(device=DEVICE).manual_seed(42)

print("Diffusion ready (Colab optimized).")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion ready (Colab optimized).


In [ ]:
def diffusion_refine(frame):
    small = cv2.resize(frame, (640, 360))

    pil = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))

    out = pipe(
        prompt="",
        image=pil,
        strength=0.2,
        guidance_scale=1.0,
        generator=generator
    ).images[0]

    out = cv2.cvtColor(np.array(out), cv2.COLOR_RGB2BGR)

    return cv2.resize(out, (frame.shape[1], frame.shape[0]))

---
## SECTION 6 - Video Background Composite (Anti-Flicker)

This replaces the old static image composite.
Uses the background video downloaded in Section 2.
Includes 3-layer anti-flicker: morphological denoise → spatial blur → temporal EMA.

In [ ]:
# =========================================================
# SETTINGS
# =========================================================
SRC_VIDEO   = "/content/src.mp4"          # your foreground subject video
BG_VIDEO    = "/content/bg.mp4"           # background video (downloaded in Section 2)
FGR_DIR     = "/content/output/fgr"       # segmented foreground frames
PHA_DIR     = "/content/output/pha"       # alpha matte frames
OUT_VIDEO   = "/content/final_output.mp4" # final output

USE_DIFFUSION       = False  # Set True to apply diffusion_refine (Section 5 must be run)
DIFFUSION_EVERY_N   = 2      # apply diffusion every N frames (2 = every other frame)

# ── Anti-flicker controls ─────────────────────────────────
ALPHA_EMA_DECAY     = 0.65   # temporal smoothing (0.5=fast/jittery, 0.85=smooth/laggy)
ALPHA_BLUR_KSIZE    = 7      # spatial edge blur (odd number, 0=off)
MORPH_KSIZE         = 5      # mask denoise kernel (0=off)

# ── Encode settings ───────────────────────────────────────
BG_LOOP             = True   # loop BG video if shorter than source
ENCODE_CRF          = 18     # 0=lossless, 23=default, 18=near-lossless
ENCODE_PRESET       = "slow" # slow=better compression, fast=quicker encode

print("Settings loaded.")

Settings loaded.


In [ ]:
# =========================================================
# HELPERS
# =========================================================
import cv2
import numpy as np
import subprocess
import shlex
import gc
import torch
from pathlib import Path
from tqdm import tqdm


def open_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open: {path}")
    return cap


def read_bg_frame(cap_bg, src_w, src_h, loop=True):
    """Read one BG frame, resize to match source, loop if exhausted."""
    ret, frame = cap_bg.read()
    if not ret:
        if loop:
            cap_bg.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap_bg.read()
        if not ret:
            return None
    if frame.shape[1] != src_w or frame.shape[0] != src_h:
        frame = cv2.resize(frame, (src_w, src_h), interpolation=cv2.INTER_LANCZOS4)
    return frame


class AlphaSmoother:
    """
    3-layer anti-flicker for alpha masks:
      1. Morphological open+close  — removes salt-and-pepper noise
      2. Spatial Gaussian blur     — softens aliased edges
      3. Temporal EMA              — kills inter-frame flicker
    """
    def __init__(self, ema_decay=0.65, blur_ksize=7, morph_ksize=5):
        self.ema_decay   = ema_decay
        self.blur_ksize  = blur_ksize | 1  # ensure odd
        self.morph_ksize = morph_ksize
        self._ema        = None
        if morph_ksize > 0:
            self._kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE, (morph_ksize, morph_ksize)
            )
        else:
            self._kernel = None

    def smooth(self, pha_uint8):
        """Input: (H,W) uint8. Output: (H,W) float32 in [0,1]."""
        pha = pha_uint8.astype(np.float32) / 255.0

        # Step 1 — morphological denoise
        if self._kernel is not None:
            pha_u8 = (pha * 255).astype(np.uint8)
            pha_u8 = cv2.morphologyEx(pha_u8, cv2.MORPH_OPEN,  self._kernel)
            pha_u8 = cv2.morphologyEx(pha_u8, cv2.MORPH_CLOSE, self._kernel)
            pha    = pha_u8.astype(np.float32) / 255.0

        # Step 2 — spatial blur
        if self.blur_ksize > 1:
            pha = cv2.GaussianBlur(pha, (self.blur_ksize, self.blur_ksize), 0)

        # Step 3 — temporal EMA
        if self._ema is None:
            self._ema = pha
        else:
            self._ema = self.ema_decay * self._ema + (1.0 - self.ema_decay) * pha

        return self._ema


def composite_frame(fgr, pha_f32, bg):
    """Alpha-blend fgr over bg. All must be same (H,W)."""
    alpha = pha_f32[..., np.newaxis]
    out = fgr.astype(np.float32) * alpha + bg.astype(np.float32) * (1.0 - alpha)
    return np.clip(out, 0, 255).astype(np.uint8)


print("Helpers defined.")

Helpers defined.


In [ ]:
# =========================================================
# MAIN — VIDEO-TO-VIDEO COMPOSITE WITH ANTI-FLICKER
# =========================================================

fgr_dir   = Path(FGR_DIR)
pha_dir   = Path(PHA_DIR)
fgr_files = sorted(fgr_dir.glob("*.png"))
n_frames  = len(fgr_files)
print(f"Foreground frames : {n_frames}")

if n_frames == 0:
    raise RuntimeError("No .png frames found in FGR_DIR. Run Section 3 or 4 first.")

# Get resolution from first frame
sample = cv2.imread(str(fgr_files[0]))
SRC_H, SRC_W = sample.shape[:2]
print(f"Source resolution : {SRC_W}x{SRC_H}")

# Get FPS from original source video
cap_src = open_video(SRC_VIDEO)
FPS     = cap_src.get(cv2.CAP_PROP_FPS) or 30.0
cap_src.release()
print(f"FPS               : {FPS:.2f}")

# Open background video
cap_bg = open_video(BG_VIDEO)
print(f"BG video frames   : {int(cap_bg.get(cv2.CAP_PROP_FRAME_COUNT))}")

# Set up ffmpeg pipe — writes directly to MP4, no intermediate PNGs
ffmpeg_cmd = (
    f"ffmpeg -y "
    f"-f rawvideo -vcodec rawvideo -s {SRC_W}x{SRC_H} "
    f"-pix_fmt bgr24 -r {FPS} -i pipe:0 "
    f"-c:v libx264 -crf {ENCODE_CRF} -preset {ENCODE_PRESET} "
    f"-pix_fmt yuv420p {OUT_VIDEO}"
)
ffmpeg_proc = subprocess.Popen(
    shlex.split(ffmpeg_cmd),
    stdin=subprocess.PIPE,
    stderr=subprocess.DEVNULL
)

smoother = AlphaSmoother(
    ema_decay   = ALPHA_EMA_DECAY,
    blur_ksize  = ALPHA_BLUR_KSIZE,
    morph_ksize = MORPH_KSIZE,
)

print("\nCompositing frames…")
try:
    for i, fgr_path in enumerate(tqdm(fgr_files, unit="fr", dynamic_ncols=True)):
        pha_path = pha_dir / fgr_path.name

        fgr = cv2.imread(str(fgr_path))
        pha = cv2.imread(str(pha_path), cv2.IMREAD_GRAYSCALE)

        if fgr is None or pha is None:
            print(f"  ⚠ Missing frame {i}, skipping")
            continue

        # Smooth alpha — removes flicker & jitter
        pha_smooth = smoother.smooth(pha)

        # Read one background frame (loops if BG is shorter)
        bg = read_bg_frame(cap_bg, SRC_W, SRC_H, loop=BG_LOOP)
        if bg is None:
            print("  ⚠ BG video exhausted (BG_LOOP=False). Stopping.")
            break

        # Composite
        out_frame = composite_frame(fgr, pha_smooth, bg)

        # Optional diffusion refinement (from Section 5)
        if USE_DIFFUSION and i % DIFFUSION_EVERY_N == 0:
            out_frame = diffusion_refine(out_frame)

        # Write to ffmpeg pipe
        ffmpeg_proc.stdin.write(out_frame.tobytes())

        # Prevent Colab VRAM OOM on long videos
        if i % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

finally:
    ffmpeg_proc.stdin.close()
    ffmpeg_proc.wait()
    cap_bg.release()

print(f"\nDone! Saved to: {OUT_VIDEO}")

Foreground frames : 384
Source resolution : 3840x2160
FPS               : 29.97
BG video frames   : 602

Compositing frames…



100%|██████████| 384/384 [04:35<00:00,  1.39fr/s]



✅ Done! Saved to: /content/final_output.mp4


---
## SECTION 7 — Preview & Download

In [ ]:
# Quick check — print file size
import os
size_mb = os.path.getsize(OUT_VIDEO) / 1024 / 1024
print(f"Output: {OUT_VIDEO}  ({size_mb:.1f} MB)")

# Download directly from Colab
from google.colab import files
files.download(OUT_VIDEO)

Output: /content/final_output.mp4  (64.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Anti-Flicker Tuning Guide

| Problem | Fix |
|---|---|
| Flickering mask edges | Raise `ALPHA_EMA_DECAY` → 0.75–0.85 |
| Ghosting / laggy edges on fast movement | Lower `ALPHA_EMA_DECAY` → 0.5–0.55 |
| Holes / specks inside mask | Raise `MORPH_KSIZE` → 7–9 |
| Hard aliased edges | Raise `ALPHA_BLUR_KSIZE` → 9–13 |
| BG video ends too early | Set `BG_LOOP = True` |
| Output looks compressed | Lower `ENCODE_CRF` → 16 |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
